In [19]:
import pandas as pd

df_fe = pd.read_parquet("/Users/llbm/Desktop/pcd/trab_final/data/nfl_train.parquet")
print(df_fe.shape)
print(df_fe.columns.tolist())
print(df_fe.dtypes.value_counts())
print(df_fe.isnull().sum().sum())

(28086, 347)
['PlayId', 'GameId', 'Season', 'Week', 'Down', 'Distance', 'HomeScoreBeforePlay', 'VisitorScoreBeforePlay', 'OffenseFormation', 'OffensePersonnel', 'DefensePersonnel', 'DefendersInTheBox', 'PossessionTeam', 'FieldPosition', 'AbsYardLine', 'GameClockPct', 'rusher_X_rel', 'rusher_Y_rel', 'rusher_S', 'rusher_A', 'rusher_Dis', 'rusher_Dir_sin', 'rusher_Dir_cos', 'rusher_Ori_sin', 'rusher_Ori_cos', 'rusher_Sx', 'rusher_Sy', 'rusher_PlayerHeight', 'rusher_PlayerWeight', 'rusher_PlayerAge', 'rusher_Position', 'off1_X_rel', 'off1_Y_rel', 'off1_S', 'off1_A', 'off1_Dis', 'off1_Dir_sin', 'off1_Dir_cos', 'off1_Ori_sin', 'off1_Ori_cos', 'off1_Sx', 'off1_Sy', 'off1_PlayerHeight', 'off1_PlayerWeight', 'off1_PlayerAge', 'off1_Position', 'off2_X_rel', 'off2_Y_rel', 'off2_S', 'off2_A', 'off2_Dis', 'off2_Dir_sin', 'off2_Dir_cos', 'off2_Ori_sin', 'off2_Ori_cos', 'off2_Sx', 'off2_Sy', 'off2_PlayerHeight', 'off2_PlayerWeight', 'off2_PlayerAge', 'off2_Position', 'off3_X_rel', 'off3_Y_rel', 'off3

In [20]:
df_test = pd.read_parquet("/Users/llbm/Desktop/pcd/trab_final/data/nfl_test.parquet")
print(df_test.shape)
print('Yards' in df_test.columns)
print(df_test['Season'].value_counts())

(2921, 347)
True
Season
2017.0    1518
2018.0    1403
Name: count, dtype: int64


In [21]:
pos_cols = [c for c in df_fe.columns if 'Position' in c]
all_pos = pd.concat([df_fe[c] for c in pos_cols]).dropna().unique()
print(sorted(all_pos))
print(f"Total: {len(all_pos)}")

[np.float32(0.0), np.float32(1.0), np.float32(2.0), np.float32(3.0), np.float32(4.0), np.float32(5.0), np.float32(6.0), np.float32(7.0), np.float32(8.0), np.float32(9.0), np.float32(10.0), np.float32(11.0), np.float32(12.0), np.float32(13.0), np.float32(14.0), np.float32(15.0), np.float32(16.0), np.float32(17.0), np.float32(18.0), np.float32(19.0), np.float32(20.0), np.float32(21.0), np.float32(22.0), np.float32(23.0), np.float32(24.0), np.float32(25.0), np.float32(26.0), np.float32(27.0), np.float32(28.0), np.float32(29.0), np.float32(30.0), np.float32(31.0), np.float32(32.0)]
Total: 33


Bagging was explored as a natural variance reduction strategy after training a single MLP model. The Zoo team (competition's winner) reported a significant performance imprevement doing so. The intuition is that models trained with different random initializations make errors in slightly different directions, and averaging their predictions tends to cancel out some of this noise. We trained 7 models with the same architecture and hyperparameters, only varying the seed, which changes both the initialization of the weights and the order in which the data is presented, but that showed to be insuficient. The aggregation uses an average weighted by the individual CRPS of each model: models that performed better in validation receive greater weight in the final CDF.

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle

In [23]:
def prepare_data(df_train, df_val):
    drop_cols = ['PlayId', 'GameId', 'Yards']
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    pos_cols = [c for c in feature_cols if 'Position' in c]
    le = LabelEncoder()
    all_pos_values = pd.concat([df_train[c] for c in pos_cols]).fillna('UNK')
    le.fit(all_pos_values)
    df_train = df_train.copy()
    df_val = df_val.copy()

    for col in pos_cols:
        df_train[col] = le.transform(df_train[col].fillna('UNK'))
        df_val[col] = df_val[col].fillna('UNK').apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else 0)

    df_train = df_train.fillna(0.0)
    df_val = df_val.fillna(0.0)
    X_train = df_train[feature_cols].values.astype(np.float32)
    X_val = df_val[feature_cols].values.astype(np.float32)
    y_train = df_train['Yards'].values.astype(np.float32)
    y_val = df_val['Yards'].values.astype(np.float32)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    return X_train, y_train, X_val, y_val, scaler, le, feature_cols

In [24]:
#same architect modules i've been using so far
class PlayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [25]:
class CRPSLoss(nn.Module):
    def __init__(self, min_yard=-99, max_yard=99, clip_min=-30, clip_max=50):
        super().__init__()
        self.clip_min = clip_min
        self.clip_max = clip_max
        self.register_buffer("y_bins", torch.arange(min_yard, max_yard + 1, dtype=torch.float32))

    def forward(self, cdf_pred, y_true):
        if self.training:
            y_true = torch.clamp(y_true, self.clip_min, self.clip_max)
        cdf_true = (self.y_bins[None, :] >= y_true[:, None]).float()
        return torch.mean((cdf_pred - cdf_true) ** 2)

In [26]:
#this MLP follows a diff pattern than most: instead of progressively reduzing hidden dimensions from input to output layer
#it first expands information into larger hidden layers aiming at solidifying the (multiple) relations extracted and individualizing signals before collapisng into bins
class MLP(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 199),)

    def forward(self, x):
        logits = self.net(x)
        pmf = F.softmax(logits, dim=-1)
        cdf = torch.cumsum(pmf, dim=-1)
        return torch.clamp(cdf, 0.0, 1.0)

In [27]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        cdf_pred = model(x_batch)
        loss = criterion(cdf_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

In [28]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            cdf_pred = model(x_batch)
            loss = criterion(cdf_pred, y_batch)
            total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

In [29]:
def predict_cdfs(model, loader, device):
    model.eval()
    all_cdfs = []
    with torch.no_grad():
        for x_batch, _ in loader:
            cdfs = model(x_batch.to(device)).cpu().numpy()
            all_cdfs.append(cdfs)
    return np.vstack(all_cdfs)  #[N, 199]

In [30]:
def crps_numpy(cdfs, y_true):
    y_bins = np.arange(-99, 100, dtype=np.float32)
    cdf_true = (y_bins[None, :] >= y_true[:, None]).astype(np.float32)
    return float(np.mean((cdfs - cdf_true) ** 2))

In [31]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [32]:
df_train = pd.read_parquet("/Users/llbm/Desktop/pcd/trab_final/data/nfl_train.parquet")
df_val = pd.read_parquet("/Users/llbm/Desktop/pcd/trab_final/data/nfl_test.parquet")
print(f"Train: {len(df_train)} | Val: {len(df_val)}")
X_train, y_train, X_val, y_val, scaler, le, feature_cols = prepare_data(df_train, df_val)
print(f"Input dim: {X_train.shape[1]}")

Train: 28086 | Val: 2921
Input dim: 344


In [33]:
BATCH_SIZE = 64
EPOCHS = 75 #dobes
LEARNING_RATE = 3e-4
N_MODELS = 7
SEEDS = list(range(N_MODELS))

In [34]:
criterion = CRPSLoss().to(device)
train_loader = DataLoader(PlayDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(PlayDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
all_val_cdfs = []   #validation CDFs for each model
all_val_crps = []  #validation CRPS for each model (for weighting)
all_states= [] #weights at the best checkpoint for each model

In [35]:
for model_idx, seed in enumerate(SEEDS):
    print(f"Model {model_idx+1}/{N_MODELS} — seed {seed}")
    torch.manual_seed(seed)
    np.random.seed(seed)
    #scramble loader with dif seed for each model 
    train_loader_seed = DataLoader(
        PlayDataset(X_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed))
    model= MLP(input_dim=X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
   #CosineAnnealingLR bc i felt i needed a lr decaying smoothly
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    
    best_val = np.inf
    best_state = None
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader_seed, optimizer, criterion, device)
        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        #printing intermediate results
        if epoch % 15 == 0 or epoch == 1 or epoch == EPOCHS:
            print(f"Epoch {epoch:02d}/{EPOCHS} | LR: {current_lr:.6f} | "
                  f"Train: {train_loss:.5f} | Val: {val_loss:.5f}")

    print(f"  >> Best Val CRPS from model {model_idx+1}: {best_val:.5f}")

    model.load_state_dict(best_state)
    val_cdfs = predict_cdfs(model, val_loader, device)
    all_val_cdfs.append(val_cdfs)
    all_val_crps.append(best_val)
    all_states.append(best_state)

Model 1/7 — seed 0
Epoch 01/75 | LR: 0.000300 | Train: 0.01592 | Val: 0.01290
Epoch 15/75 | LR: 0.000272 | Train: 0.01310 | Val: 0.01254
Epoch 30/75 | LR: 0.000200 | Train: 0.01297 | Val: 0.01253
Epoch 45/75 | LR: 0.000110 | Train: 0.01281 | Val: 0.01255
Epoch 60/75 | LR: 0.000038 | Train: 0.01267 | Val: 0.01261
Epoch 75/75 | LR: 0.000010 | Train: 0.01257 | Val: 0.01263
  >> Best Val CRPS from model 1: 0.01247
Model 2/7 — seed 1
Epoch 01/75 | LR: 0.000300 | Train: 0.01586 | Val: 0.01290
Epoch 15/75 | LR: 0.000272 | Train: 0.01310 | Val: 0.01258
Epoch 30/75 | LR: 0.000200 | Train: 0.01298 | Val: 0.01257
Epoch 45/75 | LR: 0.000110 | Train: 0.01279 | Val: 0.01264
Epoch 60/75 | LR: 0.000038 | Train: 0.01255 | Val: 0.01266
Epoch 75/75 | LR: 0.000010 | Train: 0.01236 | Val: 0.01275
  >> Best Val CRPS from model 2: 0.01254
Model 3/7 — seed 2
Epoch 01/75 | LR: 0.000300 | Train: 0.01577 | Val: 0.01281
Epoch 15/75 | LR: 0.000272 | Train: 0.01311 | Val: 0.01258
Epoch 30/75 | LR: 0.000200 | Train:

In [36]:
print("Weighted performance")
#weights inversely proportional to CRPS so "better" model has greater weight
crps_arr = np.array(all_val_crps)
weights = (1.0 / crps_arr) / (1.0 / crps_arr).sum()
print("CRPS by model:")
for i, (c, w) in enumerate(zip(crps_arr, weights)):
    print(f"  Model {i+1} (seed {i}): CRPS={c:.5f} | weight={w:.4f}")
#CDF ensamble via weighted avg
ensemble_cdfs = np.zeros_like(all_val_cdfs[0])
for cdfs, w in zip(all_val_cdfs, weights):
    ensemble_cdfs += w * cdfs
ensemble_crps = crps_numpy(ensemble_cdfs, y_val)

print(f"\nAvg individual CRPS : {np.mean(crps_arr):.5f}")
print(f"Ensamble's CRPS : {ensemble_crps:.5f}")
print(f"Improvement (gap) : {np.mean(crps_arr) - ensemble_crps:.5f}")

Weighted performance
CRPS by model:
  Model 1 (seed 0): CRPS=0.01247 | weight=0.1434
  Model 2 (seed 1): CRPS=0.01254 | weight=0.1426
  Model 3 (seed 2): CRPS=0.01253 | weight=0.1427
  Model 4 (seed 3): CRPS=0.01254 | weight=0.1426
  Model 5 (seed 4): CRPS=0.01251 | weight=0.1429
  Model 6 (seed 5): CRPS=0.01251 | weight=0.1429
  Model 7 (seed 6): CRPS=0.01251 | weight=0.1429

Avg individual CRPS : 0.01252
Ensamble's CRPS : 0.01257
Improvement (gap) : -0.00005


In [37]:
#saving evth
for i, state in enumerate(all_states):
    path = f"nfl_mlp_model_{i}.pt"
    torch.save(state, path)
artifacts = {'scaler': scaler,'label_encoder': le,'feature_cols': feature_cols,
    'seeds': SEEDS,'val_crps': all_val_crps,'weights': weights.tolist(),
    'input_dim': X_train.shape[1],'ensemble_crps': ensemble_crps,}
with open("nfl_mlp_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)
print(f"\n>> CRPS FINAL: {ensemble_crps:.5f}")


>> CRPS FINAL: 0.01257


The first bagging experiment showed that models trained only with different seeds produce very similar predictions since the variance between them is too low for the ensemble to significantly outperform the best individual model. For bagging to work well, the models need to be diverse (ideally erring in different directions so that the average cancels out errors instead of reinforcing them, but as we're adopting the exact same architecture here I don't really expect much of that difference). Still we added other three sources of explicit diversity: random dropout per model, random subset of features (which showed great impact), and augmentation by flipping the Y-axis during training. The Y-flip is motivated by the symmetry of the problem, so exposing the model to mirrored versions of the data during training improves the invariance of the learned representation. We also replaced OneCycleLR with CosineAnnealingLR, which decays smoothly without the warm-up phase, more suitable given that previous experiments showed that high learning rates hinder convergence in this problem.

In [38]:
def get_y_col_indices(feature_cols):
    #these columns should not be Y flipped
    flip_keywords = ['_Y_rel', '_Sy', '_Ori_sin', '_Dir_sin']
    return [i for i, c in enumerate(feature_cols)
            if any(c.endswith(kw) for kw in flip_keywords)]

In [39]:
def get_feature_subset(n_features, frac, seed):
    #feature subset for model
    rng = np.random.RandomState(seed)
    n_subset = int(n_features * frac)
    return sorted(rng.choice(n_features, size=n_subset, replace=False))

In [40]:
class PlayDataset(Dataset):
    def __init__(self, X, y, feature_subset=None, y_flip_cols=None, is_train=True):
        #is_train= True applies Y flipping with prob 0.5. Strategy for data augmentation. it should benefit the tails in particular the upper right one since it has the fewer examples (around 1118 or smt)
        self.y_flip_cols = y_flip_cols if y_flip_cols else []
        self.is_train = is_train
        self.feature_subset = feature_subset
        if feature_subset is not None:
            X = X[:, feature_subset]
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        if feature_subset is not None:
            subset_set = {v: i for i, v in enumerate(feature_subset)}
            self.flip_idx = [subset_set[i] for i in y_flip_cols if i in subset_set]
        else:
            self.flip_idx = list(y_flip_cols)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]

        if self.is_train and torch.rand(1).item() > 0.5:
            x[self.flip_idx] = -x[self.flip_idx]

        return x, y

In [41]:
class CRPSLoss(nn.Module):
    def __init__(self, min_yard=-99, max_yard=99, clip_min=-30, clip_max=50):
        super().__init__()
        self.clip_min = clip_min
        self.clip_max = clip_max
        self.register_buffer("y_bins", torch.arange(min_yard, max_yard + 1, dtype=torch.float32))

    def forward(self, cdf_pred, y_true):
        if self.training:
            y_true = torch.clamp(y_true, self.clip_min, self.clip_max)
        cdf_true = (self.y_bins[None, :] >= y_true[:, None]).float()
        return torch.mean((cdf_pred - cdf_true) ** 2)

In [42]:
#same base architecture for each model
class MLP(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 199),)

    def forward(self, x):
        logits = self.net(x)
        pmf = F.softmax(logits, dim=-1)
        cdf = torch.cumsum(pmf, dim=-1)
        return torch.clamp(cdf, 0.0, 1.0)

In [43]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        cdf_pred = model(x_batch)
        loss = criterion(cdf_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

In [44]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss=0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            cdf_pred = model(x_batch)
            loss = criterion(cdf_pred, y_batch)
            total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

In [45]:
def predict_cdfs(model, loader, device):
    model.eval()
    all_cdfs = []
    with torch.no_grad():
        for x_batch, _ in loader:
            cdfs = model(x_batch.to(device)).cpu().numpy()
            all_cdfs.append(cdfs)
    return np.vstack(all_cdfs)

In [46]:
def crps_numpy(cdfs, y_true):
    y_bins= np.arange(-99, 100, dtype=np.float32)
    cdf_true = (y_bins[None, :] >= y_true[:, None]).astype(np.float32)
    return float(np.mean((cdfs - cdf_true) ** 2))

In [47]:
X_train, y_train, X_val, y_val, scaler, le, feature_cols = prepare_data(df_train, df_val)
n_features = X_train.shape[1]
y_flip_cols = get_y_col_indices(feature_cols)
print(f"Input dim: {n_features} | Y flipped columns: {len(y_flip_cols)}")

SEEDS = list(range(N_MODELS))
FEATURE_FRAC  = 0.53  #adding diversity here 
criterion = CRPSLoss().to(device)
#validation dataset without augmentation and with all features
#each validation model uses its own subset
val_dataset_full = PlayDataset(X_val, y_val, y_flip_cols=y_flip_cols, is_train=False)

Input dim: 344 | Y flipped columns: 88


In [48]:
all_val_cdfs = []
all_val_crps = []
all_states = []
all_subsets = []
all_dropouts = []

for model_idx, seed in enumerate(SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.RandomState(seed)
    dropout = float(rng.uniform(0.1, 0.22))
    subset = get_feature_subset(n_features, FEATURE_FRAC, seed)
    n_sub = len(subset)

    print(f"Model {model_idx+1}/{N_MODELS} | seed {seed} | dropout={dropout:.3f} | features={n_sub}/{n_features}")

    train_dataset = PlayDataset(X_train, y_train,
        feature_subset=subset,y_flip_cols=y_flip_cols,is_train=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed))
    val_dataset = PlayDataset(X_val, y_val,feature_subset=subset,
        y_flip_cols=y_flip_cols,is_train=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MLP(input_dim=n_sub, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    
    best_val = np.inf
    best_state = None
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 15 == 0 or epoch == 1 or epoch == EPOCHS:
            print(f"Epoch {epoch:02d}/{EPOCHS} | LR: {current_lr:.6f} | "
                  f"Train: {train_loss:.5f} | Val: {val_loss:.5f}")
    print(f" >> Best val CRPS from model {model_idx+1}: {best_val:.5f}")

    model.load_state_dict(best_state)
    val_cdfs = predict_cdfs(model, val_loader, device)
    all_val_cdfs.append(val_cdfs)
    all_val_crps.append(best_val)
    all_states.append(best_state)
    all_subsets.append(subset)
    all_dropouts.append(dropout)

Model 1/7 | seed 0 | dropout=0.166 | features=189/344
Epoch 01/75 | LR: 0.000300 | Train: 0.01578 | Val: 0.01283
Epoch 15/75 | LR: 0.000272 | Train: 0.01328 | Val: 0.01253
Epoch 30/75 | LR: 0.000200 | Train: 0.01320 | Val: 0.01249
Epoch 45/75 | LR: 0.000110 | Train: 0.01314 | Val: 0.01253
Epoch 60/75 | LR: 0.000038 | Train: 0.01308 | Val: 0.01248
Epoch 75/75 | LR: 0.000010 | Train: 0.01305 | Val: 0.01246
 >> Best val CRPS from model 1: 0.01246
Model 2/7 | seed 1 | dropout=0.150 | features=189/344
Epoch 01/75 | LR: 0.000300 | Train: 0.01570 | Val: 0.01296
Epoch 15/75 | LR: 0.000272 | Train: 0.01324 | Val: 0.01257
Epoch 30/75 | LR: 0.000200 | Train: 0.01316 | Val: 0.01252
Epoch 45/75 | LR: 0.000110 | Train: 0.01311 | Val: 0.01250
Epoch 60/75 | LR: 0.000038 | Train: 0.01303 | Val: 0.01249
Epoch 75/75 | LR: 0.000010 | Train: 0.01298 | Val: 0.01249
 >> Best val CRPS from model 2: 0.01248
Model 3/7 | seed 2 | dropout=0.152 | features=189/344
Epoch 01/75 | LR: 0.000300 | Train: 0.01559 | Val:

In [49]:
print("Weighted Bagging")
crps_arr = np.array(all_val_crps)
weights = (1.0 / crps_arr) / (1.0 / crps_arr).sum()
print("CRPS by model:")
for i, (c, w, d) in enumerate(zip(crps_arr, weights, all_dropouts)):
    print(f"  Model {i+1} (seed {i}): CRPS={c:.5f} | peso={w:.4f} | dropout={d:.3f}")

ensemble_cdfs = np.zeros_like(all_val_cdfs[0])
for cdfs, w in zip(all_val_cdfs, weights):
    ensemble_cdfs += w * cdfs
ensemble_crps = crps_numpy(ensemble_cdfs, y_val)

print(f"\nmean indv CRPS : {np.mean(crps_arr):.5f}")
print(f"ensamble's CRPS : {ensemble_crps:.5f}")
print(f"Improvement (gap) : {np.mean(crps_arr) - ensemble_crps:.5f}")

Weighted Bagging
CRPS by model:
  Model 1 (seed 0): CRPS=0.01246 | peso=0.1434 | dropout=0.166
  Model 2 (seed 1): CRPS=0.01248 | peso=0.1432 | dropout=0.150
  Model 3 (seed 2): CRPS=0.01238 | peso=0.1443 | dropout=0.152
  Model 4 (seed 3): CRPS=0.01244 | peso=0.1436 | dropout=0.166
  Model 5 (seed 4): CRPS=0.01294 | peso=0.1381 | dropout=0.216
  Model 6 (seed 5): CRPS=0.01245 | peso=0.1435 | dropout=0.127
  Model 7 (seed 6): CRPS=0.01241 | peso=0.1440 | dropout=0.207

mean indv CRPS : 0.01251
ensamble's CRPS : 0.01246
Improvement (gap) : 0.00004


In [51]:
for i, state in enumerate(all_states):
    torch.save(state, f"nfl_mlp_model_{i}.pt")

artifacts = {
    'scaler': scaler,
    'label_encoder' : le,
    'feature_cols': feature_cols,
    'seeds' : SEEDS,
    'val_crps': all_val_crps,
    'weights': weights.tolist(),
    'feature_subsets': all_subsets,
    'dropouts' : all_dropouts,
    'n_features': n_features,
    'y_flip_cols': y_flip_cols,
    'ensemble_crps': ensemble_crps,}
with open("nfl_mlp_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)